In [ ]:
import pandas as pd
import numpy as np

#Load the messy data
df=pd.read_csv('marketing_campaign_data_messy.csv')

print(f'Loaded dataset: {df.shape[0]} rows,{df.shape[1]} columns')

Loaded dataset: 2020 rows,12 columns


In [ ]:
df

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01 00:00:00,2023-09-26,Google Ads,20097,897,1641.0,162.0,0,NaN,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09 00:00:00,2023-02-21,Instagram,33254,1117,883.82,214.0,0,NaN,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30 00:00:00,2023-04-27,Facebook,68728,2960,4198.5,591.0,Yes,NaN,FA


**STEP 1 : HEADER CLEANING**

In [ ]:
print(df.columns.tolist())

df.columns=df.columns.str.strip().str.lower().str.replace(' ','_')

print('Fix applied')
print(df.columns.tolist())

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
Fix applied
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


**STEP 2 : TYPE CONVERSION & CURRENCY CLEANING**

In [ ]:
dirty_spend_mask =df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask,['campaign_id', 'spend']].head(3))

# Removes '$' and ',' then converts to a float
df['spend'] = df['spend'].astype(str).str.replace(r'[\$,]', '', regex=True).astype(float)

print('Fix Applied')
print(df.loc[dirty_spend_mask,['campaign_id', 'spend']].head())
df.head()

Empty DataFrame
Columns: [campaign_id, spend]
Index: []
Fix Applied
Empty DataFrame
Columns: [campaign_id, spend]
Index: []


,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,clicks,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA


**STEP 3: CATEGORICAL TYPOS (FUZZY LOGIC)**

In [ ]:
print(df['channel'].unique())

cleanup_map = {
    'Facebok': 'Facebook',
    'Insta_gram' : 'Instagram',
    'Gogle' : 'Google Ads',
    'E-mail' : 'Email',
    'N/A' : np.nan #Handling the ghost value here too
}
df['channel'] = df['channel'].replace(cleanup_map)

print('Fix Applied')
print(df['channel'].unique())

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']
Fix Applied
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan 'Tik_Tok']


**STEP 4 : HANDLING MIXED BOOOLEANS**

In [ ]:
print(df['active'].unique())

bool_map={
    'Yes':True,
    'Y':True,
    '1' : True,
    'No':False,
    'N':False,
    '0':False
    }

df['active']=df['active'].replace(bool_map).fillna(False).astype(bool)

print('Fix Applied')
print(df['active'].unique())


['Y' '0' 'No' 'True' 'Yes' '1' 'False']
Fix Applied
[ True False]


**STEP 5: DATE PARSING**

In [ ]:
print(df['start_date'].dtype)

df['start_date']=pd.to_datetime(df['start_date'],errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')

print('Fix Applied')
print(df['start_date'].dtype)


object
Fix Applied
datetime64[ns]


**Removing Duplicated Columns**

In [ ]:
df=df.loc[:, ~df.columns.duplicated()]
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA
...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA


**STEP 5 : LOGICAL INTEGRITY CLICKS vs IMPRESSIONS**

In [ ]:
impossible_mask = df['clicks'] > df['impressions']
print(df.loc[impossible_mask,['campaign_id', 'clicks', 'impressions']].head(3))

Empty DataFrame
Columns: [campaign_id, clicks, impressions]
Index: []


**STEP 7: LOGICAL INTEGRITY (TIME TRAVEL)**

In [ ]:
time_travel_mask = df['end_date'] < df['start_date']
print(df.loc[time_travel_mask,['campaign_id','start_date','end_date']].head(5))

df.loc[time_travel_mask,'end_date']= df.loc[time_travel_mask,'start_date'] + pd.Timedelta(days=30)

print('Fix Applied')
print(df.loc[time_travel_mask,['campaign_id','start_date','end_date']].head(3))

    campaign_id start_date   end_date
23    CMP-00024 2023-05-06 2023-05-01
54    CMP-00055 2023-09-01 2023-08-27
71    CMP-00072 2023-02-01 2023-01-27
156   CMP-00157 2023-12-06 2023-12-01
200   CMP-00201 2023-01-11 2023-01-06
Fix Applied
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-06-05
54   CMP-00055 2023-09-01 2023-10-01
71   CMP-00072 2023-02-01 2023-03-03


STEP 8: HANDLING OUTLIERS(WINSORZING)

In [ ]:
Q1=df['spend'].quantile(0.25)
Q3=df['spend'].quantile(0.75)

IQR = Q3 -Q1
upper_limit = Q3 + (3*IQR)

outlier_mask = df['spend'] > upper_limit
print(df.loc[outlier_mask,['campaign_id','spend']].head(3))

print('Fix Applied')
df.loc[outlier_mask,'spend'] = upper_limit
print(df.loc[outlier_mask,['campaign_id','spend']].head(3))
#

     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
Fix Applied
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375


**STEP 9: STRING PARSING (FEATURE EXTRACTION)**

In [ ]:
print(df['campaign_name'].head(3))

# 1. Split the name by underscores: "Q1_Summer_Promo" -> ["Q1", "Summer", "Promo"]
# 2. Grab the second item (index 1): "Summer"
df['season'] = df['campaign_name'].str.split('_').str[1]

print('Fix Applied')
print(df[['campaign_name','season']].head(3))


0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: object
Fix Applied
         campaign_name  season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter
